In [ ]:
# Install required libraries
!pip install snowflake-connector-python pandas numpy matplotlib seaborn scikit-learn openpyxl faker

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 2.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of pyopenssl to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.4 MB/s eta 0:00:00
  Attempting uninstall: cryptography
    Found existing installation: cryptography 43.0.3
    Uninstalling cryptography-43.0.3:
      Successfully u

In [ ]:
import snowflake.connector
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from faker import Faker
print("All libraries imported successfully ✓")

All libraries imported successfully ✓


In [ ]:
# ============================================
# STEP 1: GENERATE SYNTHETIC MARKETING DATA
# ============================================

import pandas as pd
import numpy as np
from faker import Faker
import datetime

fake = Faker()
np.random.seed(42)

# --- Parameters ---
start_date = datetime.date(2023, 1, 1)
end_date   = datetime.date(2023, 12, 31)
date_range = pd.date_range(start=start_date, end=end_date, freq='D')

# --- Channels ---
channels = {
    'Paid Search' : {
        'platform'        : 'Google Ads',
        'avg_daily_spend' : 500,
        'ctr'             : 0.045,
        'conv_rate'       : 0.032,
        'avg_order_value' : 120
    },
    'Paid Social' : {
        'platform'        : 'Meta Ads',
        'avg_daily_spend' : 400,
        'ctr'             : 0.028,
        'conv_rate'       : 0.018,
        'avg_order_value' : 95
    },
    'Email' : {
        'platform'        : 'Salesforce MC',
        'avg_daily_spend' : 100,
        'ctr'             : 0.062,
        'conv_rate'       : 0.045,
        'avg_order_value' : 140
    },
    'Organic Search' : {
        'platform'        : 'Google Analytics',
        'avg_daily_spend' : 0,
        'ctr'             : 0.035,
        'conv_rate'       : 0.028,
        'avg_order_value' : 110
    },
    'Display' : {
        'platform'        : 'Google Ads',
        'avg_daily_spend' : 250,
        'ctr'             : 0.008,
        'conv_rate'       : 0.008,
        'avg_order_value' : 85
    },
    'Referral' : {
        'platform'        : 'Google Analytics',
        'avg_daily_spend' : 0,
        'ctr'             : 0.042,
        'conv_rate'       : 0.035,
        'avg_order_value' : 105
    }
}

# --- Campaigns ---
campaigns = [
    {'Campaign_ID': 'CMP001', 'Campaign_Name': 'Q1 Brand Awareness',    'Quarter': 'Q1'},
    {'Campaign_ID': 'CMP002', 'Campaign_Name': 'Spring Promo',          'Quarter': 'Q2'},
    {'Campaign_ID': 'CMP003', 'Campaign_Name': 'Summer Sale Push',      'Quarter': 'Q3'},
    {'Campaign_ID': 'CMP004', 'Campaign_Name': 'Q4 Holiday Campaign',   'Quarter': 'Q4'},
]

campaigns_df = pd.DataFrame(campaigns)

# --- Generate daily performance data ---
records = []

for date in date_range:
    # Assign campaign by quarter
    month = date.month
    if month <= 3:
        campaign = campaigns[0]
    elif month <= 6:
        campaign = campaigns[1]
    elif month <= 9:
        campaign = campaigns[2]
    else:
        campaign = campaigns[3]

    # Seasonality multiplier
    seasonality = {
        1: 0.85, 2: 0.88, 3: 0.95,
        4: 1.00, 5: 1.05, 6: 1.10,
        7: 1.08, 8: 1.05, 9: 1.00,
        10: 1.10, 11: 1.25, 12: 1.40
    }[month]

    for channel, params in channels.items():
        # Add realistic noise
        noise = np.random.uniform(0.85, 1.15)

        spend       = round(params['avg_daily_spend'] * seasonality * noise, 2)
        impressions = int(np.random.uniform(5000, 50000) * seasonality * noise)
        clicks      = int(impressions * params['ctr'] * noise)
        conversions = int(clicks * params['conv_rate'] * noise)
        revenue     = round(conversions * params['avg_order_value'] * noise, 2)
        new_customers = int(conversions * np.random.uniform(0.3, 0.6))

        # Derived metrics
        ctr  = round(clicks / impressions * 100, 4) if impressions > 0 else 0
        cpc  = round(spend / clicks, 4)             if clicks > 0 else 0
        cpm  = round(spend / impressions * 1000, 4) if impressions > 0 else 0
        roas = round(revenue / spend, 4)             if spend > 0 else 0
        cac  = round(spend / new_customers, 4)       if new_customers > 0 else 0
        conv_rate = round(conversions / clicks * 100, 4) if clicks > 0 else 0

        records.append({
            'Date'          : date.date(),
            'Campaign_ID'   : campaign['Campaign_ID'],
            'Campaign_Name' : campaign['Campaign_Name'],
            'Quarter'       : campaign['Quarter'],
            'Channel'       : channel,
            'Platform'      : params['platform'],
            'Spend'         : spend,
            'Impressions'   : impressions,
            'Clicks'        : clicks,
            'Conversions'   : conversions,
            'Revenue'       : revenue,
            'New_Customers' : new_customers,
            'CTR'           : ctr,
            'CPC'           : cpc,
            'CPM'           : cpm,
            'ROAS'          : roas,
            'CAC'           : cac,
            'Conv_Rate'     : conv_rate
        })

marketing_data = pd.DataFrame(records)

print("=== Dataset Overview ===")
print(f"Total rows        : {marketing_data.shape[0]:,}")
print(f"Date range        : {marketing_data['Date'].min()} to {marketing_data['Date'].max()}")
print(f"Channels          : {marketing_data['Channel'].nunique()}")
print(f"Campaigns         : {marketing_data['Campaign_Name'].nunique()}")
print(f"Total Spend       : ${marketing_data['Spend'].sum():,.2f}")
print(f"Total Revenue     : ${marketing_data['Revenue'].sum():,.2f}")
print(f"Overall ROAS      : {round(marketing_data['Revenue'].sum() / marketing_data['Spend'].sum(), 2)}x")
print(f"Total Conversions : {marketing_data['Conversions'].sum():,}")
print(f"\nSample rows:")
print(marketing_data.head(3).to_string())
print("\nStep 1 complete ✓")

=== Dataset Overview ===
Total rows        : 2,190
Date range        : 2023-01-01 to 2023-12-31
Channels          : 6
Campaigns         : 4
Total Spend       : $484,851.13
Total Revenue     : $9,543,415.63
Overall ROAS      : 19.68x
Total Conversions : 77,027

Sample rows:
         Date Campaign_ID       Campaign_Name Quarter      Channel       Platform   Spend  Impressions  Clicks  Conversions  Revenue  New_Customers     CTR     CPC      CPM      ROAS       CAC  Conv_Rate
0  2023-01-01      CMP001  Q1 Brand Awareness      Q1  Paid Search     Google Ads  409.00        39086    1692           52  6005.14             27  4.3289  0.2417  10.4641   14.6825   15.1481     3.0733
1  2023-01-01      CMP001  Q1 Brand Awareness      Q1  Paid Social       Meta Ads  350.06        10520     303            5   489.06              1  2.8802  1.1553  33.2757    1.3971  350.0600     1.6502
2  2023-01-01      CMP001  Q1 Brand Awareness      Q1        Email  Salesforce MC   73.73        32425    1743    

In [ ]:
marketing_data["Campaign_Name"].unique()

array(['Q1 Brand Awareness', 'Spring Promo', 'Summer Sale Push',
       'Q4 Holiday Campaign'], dtype=object)

In [ ]:
# ============================================
# STEP 2: BUILD STAR SCHEMA & LOAD TO SNOWFLAKE
# ============================================

import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

# --- Build Dimension Tables ---

# DIM_DATE
dim_date = pd.DataFrame({
    'Date'        : pd.date_range('2023-01-01', '2023-12-31', freq='D')
})
dim_date['Date']        = dim_date['Date'].dt.date
dim_date['Year']        = pd.to_datetime(dim_date['Date']).dt.year
dim_date['Quarter']     = pd.to_datetime(dim_date['Date']).dt.quarter.map(
                          {1:'Q1', 2:'Q2', 3:'Q3', 4:'Q4'})
dim_date['Month']       = pd.to_datetime(dim_date['Date']).dt.month
dim_date['Month_Name']  = pd.to_datetime(dim_date['Date']).dt.strftime('%B')
dim_date['Week']        = pd.to_datetime(dim_date['Date']).dt.isocalendar().week.astype(int)
dim_date['Day_Name']    = pd.to_datetime(dim_date['Date']).dt.day_name()
dim_date['Is_Weekend']  = pd.to_datetime(dim_date['Date']).dt.dayofweek >= 5

# DIM_CHANNEL
dim_channel = pd.DataFrame({
    'Channel'     : list(channels.keys()),
    'Platform'    : [v['platform']        for v in channels.values()],
    'Is_Paid'     : [v['avg_daily_spend'] > 0 for v in channels.values()],
    'Avg_CPC_Benchmark'  : [0.50, 0.75, 0.10, 0.00, 0.25, 0.00],
    'Avg_ROAS_Benchmark' : [8.0,  5.0,  15.0, 0.0,  3.0,  0.0]
})

# DIM_CAMPAIGN
dim_campaign = campaigns_df.copy()
dim_campaign['Start_Date'] = ['2023-01-01', '2023-04-01',
                               '2023-07-01', '2023-10-01']
dim_campaign['End_Date']   = ['2023-03-31', '2023-06-30',
                               '2023-09-30', '2023-12-31']
dim_campaign['Budget']     = [100000, 120000, 130000, 150000]

# FACT TABLE
fact_table = marketing_data[[
    'Date', 'Campaign_ID', 'Channel',
    'Spend', 'Impressions', 'Clicks',
    'Conversions', 'Revenue', 'New_Customers',
    'CTR', 'CPC', 'CPM', 'ROAS', 'CAC', 'Conv_Rate'
]].copy()

print("=== Schema Summary ===")
print(f"DIM_DATE      : {dim_date.shape[0]:,} rows, {dim_date.shape[1]} columns")
print(f"DIM_CHANNEL   : {dim_channel.shape[0]:,} rows, {dim_channel.shape[1]} columns")
print(f"DIM_CAMPAIGN  : {dim_campaign.shape[0]:,} rows, {dim_campaign.shape[1]} columns")
print(f"FACT_TABLE    : {fact_table.shape[0]:,} rows, {fact_table.shape[1]} columns")

# --- Connect to Snowflake ---
conn = snowflake.connector.connect(
    user      = 'ATAHERI',
    password  = 'Az@r822002822002',
    account   = 'NMACXBT-NT19505',
    warehouse = 'COMPUTE_WH'
)

cursor = conn.cursor()

# --- Create new database ---
cursor.execute("CREATE DATABASE IF NOT EXISTS MARKETING_PIPELINE")
cursor.execute("USE DATABASE MARKETING_PIPELINE")
cursor.execute("CREATE SCHEMA IF NOT EXISTS PUBLIC")
cursor.execute("USE SCHEMA PUBLIC")
print("\nDatabase ready ✓")

# --- Create Tables ---
cursor.execute("""
    CREATE OR REPLACE TABLE DIM_DATE (
        DATE        DATE,
        YEAR        INTEGER,
        QUARTER     VARCHAR,
        MONTH       INTEGER,
        MONTH_NAME  VARCHAR,
        WEEK        INTEGER,
        DAY_NAME    VARCHAR,
        IS_WEEKEND  BOOLEAN
    )
""")

cursor.execute("""
    CREATE OR REPLACE TABLE DIM_CHANNEL (
        CHANNEL             VARCHAR,
        PLATFORM            VARCHAR,
        IS_PAID             BOOLEAN,
        AVG_CPC_BENCHMARK   FLOAT,
        AVG_ROAS_BENCHMARK  FLOAT
    )
""")

cursor.execute("""
    CREATE OR REPLACE TABLE DIM_CAMPAIGN (
        CAMPAIGN_ID     VARCHAR,
        CAMPAIGN_NAME   VARCHAR,
        QUARTER         VARCHAR,
        START_DATE      DATE,
        END_DATE        DATE,
        BUDGET          FLOAT
    )
""")

cursor.execute("""
    CREATE OR REPLACE TABLE FACT_CAMPAIGN_PERFORMANCE (
        DATE            DATE,
        CAMPAIGN_ID     VARCHAR,
        CHANNEL         VARCHAR,
        SPEND           FLOAT,
        IMPRESSIONS     INTEGER,
        CLICKS          INTEGER,
        CONVERSIONS     INTEGER,
        REVENUE         FLOAT,
        NEW_CUSTOMERS   INTEGER,
        CTR             FLOAT,
        CPC             FLOAT,
        CPM             FLOAT,
        ROAS            FLOAT,
        CAC             FLOAT,
        CONV_RATE       FLOAT
    )
""")
print("Tables created ✓")

# --- Prepare uploads ---
def prep_df(df):
    df = df.copy().reset_index(drop=True)
    df.columns = df.columns.str.upper()
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str)
    for col in df.select_dtypes(include='bool').columns:
        df[col] = df[col].astype(bool)
    return df

dim_date_up     = prep_df(dim_date)
dim_channel_up  = prep_df(dim_channel)
dim_campaign_up = prep_df(dim_campaign)
fact_up         = prep_df(fact_table)

# Fix date columns
dim_date_up['DATE']              = pd.to_datetime(dim_date_up['DATE']).dt.strftime('%Y-%m-%d')
dim_campaign_up['START_DATE']    = pd.to_datetime(dim_campaign_up['START_DATE']).dt.strftime('%Y-%m-%d')
dim_campaign_up['END_DATE']      = pd.to_datetime(dim_campaign_up['END_DATE']).dt.strftime('%Y-%m-%d')
fact_up['DATE']                  = pd.to_datetime(fact_up['DATE']).dt.strftime('%Y-%m-%d')

# --- Upload ---
write_pandas(conn, dim_date_up,     'DIM_DATE')
print("DIM_DATE loaded ✓")

write_pandas(conn, dim_channel_up,  'DIM_CHANNEL')
print("DIM_CHANNEL loaded ✓")

write_pandas(conn, dim_campaign_up, 'DIM_CAMPAIGN')
print("DIM_CAMPAIGN loaded ✓")

write_pandas(conn, fact_up,         'FACT_CAMPAIGN_PERFORMANCE')
print("FACT_CAMPAIGN_PERFORMANCE loaded ✓")

# --- Validate ---
print("\n=== Snowflake Validation ===")
for table in ['DIM_DATE', 'DIM_CHANNEL',
              'DIM_CAMPAIGN', 'FACT_CAMPAIGN_PERFORMANCE']:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    count = cursor.fetchone()[0]
    print(f"  {table:30s}: {count:,} rows")

cursor.close()
conn.close()
print("\nAll data loaded to Snowflake successfully ✓")

=== Schema Summary ===
DIM_DATE      : 365 rows, 8 columns
DIM_CHANNEL   : 6 rows, 5 columns
DIM_CAMPAIGN  : 4 rows, 6 columns
FACT_TABLE    : 2,190 rows, 15 columns

Database ready ✓
Tables created ✓
DIM_DATE loaded ✓
DIM_CHANNEL loaded ✓
DIM_CAMPAIGN loaded ✓
FACT_CAMPAIGN_PERFORMANCE loaded ✓

=== Snowflake Validation ===
  DIM_DATE                      : 365 rows
  DIM_CHANNEL                   : 6 rows
  DIM_CAMPAIGN                  : 4 rows
  FACT_CAMPAIGN_PERFORMANCE     : 2,190 rows

All data loaded to Snowflake successfully ✓


In [ ]:
dim_campaign_up

,CAMPAIGN_ID,CAMPAIGN_NAME,QUARTER,START_DATE,END_DATE,BUDGET
0,CMP001,Q1 Brand Awareness,Q1,2023-01-01,2023-03-31,100000
1,CMP002,Spring Promo,Q2,2023-04-01,2023-06-30,120000
2,CMP003,Summer Sale Push,Q3,2023-07-01,2023-09-30,130000
3,CMP004,Q4 Holiday Campaign,Q4,2023-10-01,2023-12-31,150000


In [ ]:
dim_channel_up

,CHANNEL,PLATFORM,IS_PAID,AVG_CPC_BENCHMARK,AVG_ROAS_BENCHMARK
0,Paid Search,Google Ads,True,0.50,8.0
1,Paid Social,Meta Ads,True,0.75,5.0
2,Email,Salesforce MC,True,0.10,15.0
3,Organic Search,Google Analytics,False,0.00,0.0
4,Display,Google Ads,True,0.25,3.0
5,Referral,Google Analytics,False,0.00,0.0


In [ ]:
dim_date_up

,DATE,YEAR,QUARTER,MONTH,MONTH_NAME,WEEK,DAY_NAME,IS_WEEKEND
0,2023-01-01,2023,Q1,1,January,52,Sunday,True
1,2023-01-02,2023,Q1,1,January,1,Monday,False
2,2023-01-03,2023,Q1,1,January,1,Tuesday,False
3,2023-01-04,2023,Q1,1,January,1,Wednesday,False
4,2023-01-05,2023,Q1,1,January,1,Thursday,False
...,...,...,...,...,...,...,...,...
360,2023-12-27,2023,Q4,12,December,52,Wednesday,False
361,2023-12-28,2023,Q4,12,December,52,Thursday,False
362,2023-12-29,2023,Q4,12,December,52,Friday,False
363,2023-12-30,2023,Q4,12,December,52,Saturday,True


In [ ]:
fact_up

,DATE,CAMPAIGN_ID,CHANNEL,SPEND,IMPRESSIONS,CLICKS,CONVERSIONS,REVENUE,NEW_CUSTOMERS,CTR,CPC,CPM,ROAS,CAC,CONV_RATE
0,2023-01-01,CMP001,Paid Search,409.00,39086,1692,52,6005.14,27,4.3289,0.2417,10.4641,14.6825,15.1481,3.0733
1,2023-01-01,CMP001,Paid Social,350.06,10520,303,5,489.06,1,2.8802,1.1553,33.2757,1.3971,350.0600,1.6502
2,2023-01-01,CMP001,Email,73.73,32425,1743,68,8257.89,32,5.3755,0.0423,2.2739,112.0018,2.3041,3.9013
3,2023-01-01,CMP001,Organic Search,0.00,5351,198,5,584.33,2,3.7002,0.0000,0.0000,0.0000,0.0000,2.5253
4,2023-01-01,CMP001,Display,233.69,13605,119,1,93.48,0,0.8747,1.9638,17.1768,0.4000,0.0000,0.8403
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2185,2023-12-31,CMP004,Paid Social,595.79,31998,953,18,1819.28,5,2.9783,0.6252,18.6196,3.0536,119.1580,1.8888
2186,2023-12-31,CMP004,Email,150.71,72313,4826,233,35114.77,86,6.6738,0.0312,2.0841,232.9956,1.7524,4.8280
2187,2023-12-31,CMP004,Organic Search,0.00,17596,583,15,1562.64,7,3.3133,0.0000,0.0000,0.0000,0.0000,2.5729
2188,2023-12-31,CMP004,Display,313.76,26391,189,1,76.20,0,0.7162,1.6601,11.8889,0.2429,0.0000,0.5291


In [ ]:
# ============================================
# STEP 3: CHANNEL PERFORMANCE ANALYSIS
# ============================================

# --- 3A: Overall Channel Summary ---
channel_summary = marketing_data.groupby('Channel').agg(
    Total_Spend     = ('Spend', 'sum'),
    Total_Revenue   = ('Revenue', 'sum'),
    Total_Clicks    = ('Clicks', 'sum'),
    Total_Impressions = ('Impressions', 'sum'),
    Total_Conversions = ('Conversions', 'sum'),
    Total_New_Customers = ('New_Customers', 'sum'),
    Avg_CTR         = ('CTR', 'mean'),
    Avg_CPC         = ('CPC', 'mean'),
    Avg_ROAS        = ('ROAS', 'mean'),
    Avg_CAC         = ('CAC', 'mean'),
    Avg_Conv_Rate   = ('Conv_Rate', 'mean')
).round(2).reset_index()

channel_summary['Revenue_Share_%'] = (
    channel_summary['Total_Revenue'] /
    channel_summary['Total_Revenue'].sum() * 100
).round(2)

channel_summary['Spend_Share_%'] = (
    channel_summary['Total_Spend'] /
    channel_summary['Total_Spend'].sum() * 100
).round(2)

print("=== Channel Performance Summary ===")
print(channel_summary[[
    'Channel', 'Total_Spend', 'Total_Revenue',
    'Avg_ROAS', 'Avg_CAC', 'Avg_CTR',
    'Revenue_Share_%', 'Spend_Share_%'
]].to_string(index=False))

# --- 3B: Monthly Trend by Channel ---
marketing_data['Month'] = pd.to_datetime(
    marketing_data['Date']).dt.month
marketing_data['Month_Name'] = pd.to_datetime(
    marketing_data['Date']).dt.strftime('%b')

monthly_trend = marketing_data.groupby(
    ['Month', 'Month_Name', 'Channel']).agg(
    Total_Spend   = ('Spend', 'sum'),
    Total_Revenue = ('Revenue', 'sum'),
    Avg_ROAS      = ('ROAS', 'mean')
).round(2).reset_index()

# --- 3C: Quarter over Quarter ---
qoq = marketing_data.groupby(
    ['Quarter', 'Channel']).agg(
    Total_Spend     = ('Spend', 'sum'),
    Total_Revenue   = ('Revenue', 'sum'),
    Total_Conversions = ('Conversions', 'sum'),
    Avg_ROAS        = ('ROAS', 'mean'),
    Avg_CAC         = ('CAC', 'mean')
).round(2).reset_index()

print("\n=== Quarter over Quarter by Channel ===")
print(qoq.to_string(index=False))

# --- 3D: Best & Worst Performing Channels ---
print("\n=== Top 3 Channels by ROAS ===")
print(channel_summary.nlargest(3, 'Avg_ROAS')[
    ['Channel', 'Avg_ROAS', 'Total_Revenue',
     'Total_Spend']].to_string(index=False))

print("\n=== Bottom 3 Channels by ROAS ===")
print(channel_summary.nsmallest(3, 'Avg_ROAS')[
    ['Channel', 'Avg_ROAS', 'Total_Revenue',
     'Total_Spend']].to_string(index=False))

# --- 3E: Budget Efficiency ---
print("\n=== Budget Efficiency (Spend Share vs Revenue Share) ===")
print(channel_summary[[
    'Channel', 'Spend_Share_%',
    'Revenue_Share_%']].assign(
    Efficiency_Gap = lambda x:
    (x['Revenue_Share_%'] - x['Spend_Share_%']).round(2)
).to_string(index=False))

print("\nStep 3 complete ✓")

=== Channel Performance Summary ===
       Channel  Total_Spend  Total_Revenue  Avg_ROAS  Avg_CAC  Avg_CTR  Revenue_Share_%  Spend_Share_%
       Display     96931.22       44139.54      0.44    58.68     0.80             0.46          19.99
         Email     38512.33     4141234.66    104.65     5.02     6.16            43.39           7.94
Organic Search         0.00     1204006.66      0.00     0.00     3.51            12.62           0.00
   Paid Search    194578.63     1912176.67      9.58    49.18     4.52            20.04          40.13
   Paid Social    154828.95      507011.93      3.20   122.64     2.80             5.31          31.93
      Referral         0.00     1734846.17      0.00     0.00     4.23            18.18           0.00

=== Quarter over Quarter by Channel ===
Quarter        Channel  Total_Spend  Total_Revenue  Total_Conversions  Avg_ROAS  Avg_CAC
     Q1        Display     20283.45        9517.28                108      0.46    45.17
     Q1          Email  

In [ ]:
# ============================================
# STEP 4: ANOMALY DETECTION ON SPEND & ROAS
# ============================================

# --- 4A: Rolling Average & Standard Deviation ---
marketing_data_sorted = marketing_data.sort_values(
    ['Channel', 'Date']).copy()

# Calculate 7-day rolling mean and std per channel
marketing_data_sorted['Rolling_Mean_Spend'] = (
    marketing_data_sorted.groupby('Channel')['Spend']
    .transform(lambda x: x.rolling(window=7, min_periods=1).mean())
).round(2)

marketing_data_sorted['Rolling_Std_Spend'] = (
    marketing_data_sorted.groupby('Channel')['Spend']
    .transform(lambda x: x.rolling(window=7, min_periods=1).std())
).round(2)

# --- 4B: Flag Anomalies (Z-score > 2) ---
marketing_data_sorted['Z_Score_Spend'] = (
    (marketing_data_sorted['Spend'] -
     marketing_data_sorted['Rolling_Mean_Spend']) /
    marketing_data_sorted['Rolling_Std_Spend']
).round(4)

marketing_data_sorted['Spend_Anomaly'] = (
    marketing_data_sorted['Z_Score_Spend'].abs() > 2
)

# --- 4C: ROAS Anomalies ---
marketing_data_sorted['Rolling_Mean_ROAS'] = (
    marketing_data_sorted.groupby('Channel')['ROAS']
    .transform(lambda x: x.rolling(window=7, min_periods=1).mean())
).round(2)

marketing_data_sorted['Rolling_Std_ROAS'] = (
    marketing_data_sorted.groupby('Channel')['ROAS']
    .transform(lambda x: x.rolling(window=7, min_periods=1).std())
).round(2)

marketing_data_sorted['Z_Score_ROAS'] = (
    (marketing_data_sorted['ROAS'] -
     marketing_data_sorted['Rolling_Mean_ROAS']) /
    marketing_data_sorted['Rolling_Std_ROAS']
).round(4)

marketing_data_sorted['ROAS_Anomaly'] = (
    marketing_data_sorted['Z_Score_ROAS'].abs() > 2
)

# --- 4D: Anomaly Summary ---
spend_anomalies = marketing_data_sorted[
    marketing_data_sorted['Spend_Anomaly'] == True]

roas_anomalies = marketing_data_sorted[
    marketing_data_sorted['ROAS_Anomaly'] == True]

print("=== Spend Anomalies Detected ===")
print(f"Total flagged: {len(spend_anomalies):,} records")
print(spend_anomalies[[
    'Date', 'Channel', 'Spend',
    'Rolling_Mean_Spend', 'Z_Score_Spend'
]].head(10).to_string(index=False))

print("\n=== ROAS Anomalies Detected ===")
print(f"Total flagged: {len(roas_anomalies):,} records")
print(roas_anomalies[[
    'Date', 'Channel', 'ROAS',
    'Rolling_Mean_ROAS', 'Z_Score_ROAS'
]].head(10).to_string(index=False))

print("\n=== Anomalies by Channel ===")
anomaly_summary = marketing_data_sorted.groupby('Channel').agg(
    Spend_Anomalies = ('Spend_Anomaly', 'sum'),
    ROAS_Anomalies  = ('ROAS_Anomaly', 'sum'),
    Total_Days      = ('Date', 'count')
).reset_index()

anomaly_summary['Anomaly_Rate_%'] = (
    (anomaly_summary['Spend_Anomalies'] +
     anomaly_summary['ROAS_Anomalies']) /
    (anomaly_summary['Total_Days'] * 2) * 100
).round(2)

print(anomaly_summary.to_string(index=False))
print("\nStep 4 complete ✓")

=== Spend Anomalies Detected ===
Total flagged: 10 records
      Date     Channel  Spend  Rolling_Mean_Spend  Z_Score_Spend
2023-11-01     Display 330.59              294.27         2.0672
2023-02-13       Email  93.12               82.24         2.0763
2023-03-08       Email  82.05              101.08        -2.1121
2023-10-15       Email  94.19              112.75        -2.0262
2023-11-29       Email 142.26              126.13         2.0627
2023-10-02 Paid Search 626.16              507.27         2.0756
2023-10-11 Paid Search 474.95              574.29        -2.0175
2023-03-18 Paid Social 343.63              390.62        -2.0047
2023-05-30 Paid Social 359.44              428.14        -2.0793
2023-07-03 Paid Social 376.80              455.44        -2.0159

=== ROAS Anomalies Detected ===
Total flagged: 17 records
      Date Channel     ROAS  Rolling_Mean_ROAS  Z_Score_ROAS
2023-07-04 Display   0.9444               0.18        2.1233
2023-08-26 Display   0.3238               0.6

In [ ]:
# ============================================
# STEP 4B: UPDATE SNOWFLAKE WITH ANOMALY FLAGS
# ============================================

conn = snowflake.connector.connect(
    user      = 'ATAHERI',
    password  = 'Az@r822002822002',
    account   = 'NMACXBT-NT19505',
    warehouse = 'COMPUTE_WH',
    database  = 'MARKETING_PIPELINE',
    schema    = 'PUBLIC'
)

cursor = conn.cursor()

cursor.execute("""
    CREATE OR REPLACE TABLE FACT_CAMPAIGN_PERFORMANCE (
        DATE                DATE,
        CAMPAIGN_ID         VARCHAR,
        CHANNEL             VARCHAR,
        SPEND               FLOAT,
        IMPRESSIONS         INTEGER,
        CLICKS              INTEGER,
        CONVERSIONS         INTEGER,
        REVENUE             FLOAT,
        NEW_CUSTOMERS       INTEGER,
        CTR                 FLOAT,
        CPC                 FLOAT,
        CPM                 FLOAT,
        ROAS                FLOAT,
        CAC                 FLOAT,
        CONV_RATE           FLOAT,
        SPEND_ANOMALY       BOOLEAN,
        ROAS_ANOMALY        BOOLEAN,
        Z_SCORE_SPEND       FLOAT,
        Z_SCORE_ROAS        FLOAT,
        ROLLING_MEAN_SPEND  FLOAT,
        ROLLING_MEAN_ROAS   FLOAT
    )
""")
print("Table recreated ✓")

# --- Prepare enriched fact table ---
fact_enriched = marketing_data_sorted[[
    'Date', 'Campaign_ID', 'Channel',
    'Spend', 'Impressions', 'Clicks',
    'Conversions', 'Revenue', 'New_Customers',
    'CTR', 'CPC', 'CPM', 'ROAS', 'CAC', 'Conv_Rate',
    'Spend_Anomaly', 'ROAS_Anomaly',
    'Z_Score_Spend', 'Z_Score_ROAS',
    'Rolling_Mean_Spend', 'Rolling_Mean_ROAS'
]].copy().reset_index(drop=True)

# --- Clean up ---
fact_enriched['Z_Score_Spend']      = fact_enriched['Z_Score_Spend'].fillna(0)
fact_enriched['Z_Score_ROAS']       = fact_enriched['Z_Score_ROAS'].fillna(0)
fact_enriched['Rolling_Mean_Spend'] = fact_enriched['Rolling_Mean_Spend'].fillna(0)
fact_enriched['Rolling_Mean_ROAS']  = fact_enriched['Rolling_Mean_ROAS'].fillna(0)

# --- Uppercase columns ---
fact_enriched.columns = fact_enriched.columns.str.upper()

# --- Fix types ---
fact_enriched['DATE'] = pd.to_datetime(
    fact_enriched['DATE']).dt.strftime('%Y-%m-%d')

for col in fact_enriched.select_dtypes(include='object').columns:
    fact_enriched[col] = fact_enriched[col].astype(str)

fact_enriched['SPEND_ANOMALY'] = fact_enriched[
    'SPEND_ANOMALY'].map({True: True, False: False,
                          'True': True, 'False': False})
fact_enriched['ROAS_ANOMALY']  = fact_enriched[
    'ROAS_ANOMALY'].map({True: True, False: False,
                         'True': True, 'False': False})

# --- Upload ---
write_pandas(conn, fact_enriched, 'FACT_CAMPAIGN_PERFORMANCE')
print("Fact table loaded ✓")

# --- Validate ---
cursor.execute("SELECT COUNT(*) FROM FACT_CAMPAIGN_PERFORMANCE")
print(f"Total rows      : {cursor.fetchone()[0]:,}")

cursor.execute("""
    SELECT COUNT(*) FROM FACT_CAMPAIGN_PERFORMANCE
    WHERE SPEND_ANOMALY = TRUE
""")
print(f"Spend anomalies : {cursor.fetchone()[0]}")

cursor.execute("""
    SELECT COUNT(*) FROM FACT_CAMPAIGN_PERFORMANCE
    WHERE ROAS_ANOMALY = TRUE
""")
print(f"ROAS anomalies  : {cursor.fetchone()[0]}")

cursor.close()
conn.close()
print("\nSnowflake updated successfully ✓")




Table recreated ✓
Fact table loaded ✓
Total rows      : 2,190
Spend anomalies : 10
ROAS anomalies  : 17

Snowflake updated successfully ✓
